In [1]:
import os
import sys

# VS Code'un eksik aldığı yolları manuel olarak Windows'a tanıtıyoruz
torch_lib_path = r"C:\Users\Mustafa\s_env\Lib\site-packages\torch\lib"
env_scripts_path = r"C:\Users\Mustafa\s_env\Scripts"

os.environ['PATH'] = torch_lib_path + ";" + env_scripts_path + ";" + os.environ.get('PATH', '')
os.add_dll_directory(torch_lib_path)

import torch
print("PyTorch Başarıyla Yüklendi! Versiyon:", torch.__version__)

PyTorch Başarıyla Yüklendi! Versiyon: 2.13.0+cu126


In [2]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset
from transformers import (
    AutoTokenizer, AutoModel, Trainer, TrainingArguments, EarlyStoppingCallback
)
from sklearn.preprocessing import LabelEncoder
from sklearn.utils.class_weight import compute_class_weight
import mlflow
import re

mlflow.set_tracking_uri("sqlite:///../mlflow.db")
mlflow.set_experiment("transformer_finetuning")

TASK_COLS = ["type", "queue", "category", "priority"]
MODEL_NAME = "bert-base-multilingual-cased"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

c:\Users\Mustafa\s_env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
c:\Users\Mustafa\s_env\Lib\site-packages\pydantic\_internal\_fields.py:132: UserWarning: Field "model_name" in PromptModelConfig has conflict with protected namespace "model_".

You may be able to resolve this warning by setting `model_config['protected_namespaces'] = ()`.
  warnings.warn(


Device: cuda


In [3]:
def light_clean(t):
    if not isinstance(t, str):
        return ""
    t = t.replace("\\n", " ")
    t = re.sub(r"<[^>]+>", " ", t)
    t = re.sub(r"\s+", " ", t).strip()
    return t

train_df = pd.read_json("../data/processed/train.jsonl", lines=True)
val_df = pd.read_json("../data/processed/val.jsonl", lines=True)
train_df["body_light_clean"] = train_df["body"].apply(light_clean)
val_df["body_light_clean"] = val_df["body"].apply(light_clean)

In [4]:
encoders = {}
y_train_dict, y_val_dict, class_weights = {}, {}, {}

for col in TASK_COLS:
    le = LabelEncoder()
    y_train_dict[col] = le.fit_transform(train_df[col])
    y_val_dict[col] = le.transform(val_df[col])
    encoders[col] = le
    classes = np.arange(len(le.classes_))
    w = compute_class_weight(class_weight="balanced", classes=classes, y=y_train_dict[col])
    class_weights[col] = torch.tensor(w, dtype=torch.float32).to(device)

num_classes_dict = {col: len(encoders[col].classes_) for col in TASK_COLS}

In [5]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
MAX_LENGTH = 128  # %95'lik dilim ~92 kelime, subword token biraz daha fazla ama 128 rahat kapsiyor

class TicketDataset(Dataset):
    def __init__(self, texts, y_dict, tokenizer, max_length):
        self.texts = texts
        self.y_dict = y_dict
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        enc = self.tokenizer(
            self.texts[idx], truncation=True, padding="max_length",
            max_length=self.max_length, return_tensors="pt"
        )
        item = {
            "input_ids": enc["input_ids"].squeeze(0),
            "attention_mask": enc["attention_mask"].squeeze(0),
        }
        for col in TASK_COLS:
            item[col] = torch.tensor(self.y_dict[col][idx], dtype=torch.long)
        return item

train_ds = TicketDataset(train_df["body_light_clean"].values, y_train_dict, tokenizer, MAX_LENGTH)
val_ds = TicketDataset(val_df["body_light_clean"].values, y_val_dict, tokenizer, MAX_LENGTH)

In [6]:
class MultiTaskTransformer(nn.Module):
    def __init__(self, model_name, num_classes_dict, class_weights=None):
        super().__init__()
        self.backbone = AutoModel.from_pretrained(model_name)
        hidden = self.backbone.config.hidden_size
        self.dropout = nn.Dropout(0.1)
        self.task_names = list(num_classes_dict.keys())
        # ONEMLI: ModuleDict anahtarlarinda "type" gibi task ismini DOGRUDAN kullanma
        # (nn.Module'un .type() metoduyla catisir) -> "head_" prefix'i zorunlu
        self.heads = nn.ModuleDict({f"head_{t}": nn.Linear(hidden, n) for t, n in num_classes_dict.items()})
        self.class_weights = class_weights or {}

    def forward(self, input_ids=None, attention_mask=None, **task_labels):
        outputs = self.backbone(input_ids=input_ids, attention_mask=attention_mask)
        pooled = self.dropout(outputs.last_hidden_state[:, 0])  # [CLS]
        logits = {t: self.heads[f"head_{t}"](pooled) for t in self.task_names}

        loss = None
        provided = {t: task_labels[t] for t in self.task_names if t in task_labels and task_labels[t] is not None}
        if provided:
            loss = sum(
                nn.functional.cross_entropy(logits[t], labels, weight=self.class_weights.get(t))
                for t, labels in provided.items()
            )
        # HF eval dongusu tensor/tuple bekler; siralamayi task_names ile sabitliyoruz
        logits_tuple = tuple(logits[t] for t in self.task_names)
        return {"loss": loss, "logits": logits_tuple}


class MultiTaskTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        outputs = model(**inputs)
        return (outputs["loss"], outputs) if return_outputs else outputs["loss"]

In [7]:
def compute_metrics(eval_pred):
    logits_tuple, label_ids = eval_pred
    result = {}
    for i, t in enumerate(TASK_COLS):
        preds = np.argmax(logits_tuple[i], axis=1)
        result[f"{t}_accuracy"] = (preds == label_ids[i]).mean()
    return result

In [8]:
from transformers import TrainerCallback

class MLflowCallback(TrainerCallback):
    def on_log(self, args, state, control, logs=None, **kwargs):
        if logs:
            for k, v in logs.items():
                if isinstance(v, (int, float)):
                    mlflow.log_metric(k, v, step=state.global_step)

In [9]:
os.makedirs("../models", exist_ok=True)

model = MultiTaskTransformer(MODEL_NAME, num_classes_dict, class_weights)

training_args = TrainingArguments(
    output_dir="../models/bert_multitask_checkpoints",
    per_device_train_batch_size=16,      # 6GB VRAM icin guvenli baslangic
    per_device_eval_batch_size=32,
    gradient_accumulation_steps=2,        # efektif batch = 32
    num_train_epochs=5,
    learning_rate=2e-5,
    lr_scheduler_type="linear",
    warmup_steps=100,
    fp16=True,                             # mixed precision, VRAM tasarrufu
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_steps=50,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    remove_unused_columns=False,           # KRITIK: custom label sutunlari icin sart
    label_names=TASK_COLS,                 # KRITIK: eval_loss hesaplanmasi icin sart
    report_to=[],                          # kendi MLflow callback'imizi kullaniyoruz
)

trainer = MultiTaskTrainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2), MLflowCallback()],
)

with mlflow.start_run(run_name="bert_multitask"):
    mlflow.log_param("model_name", MODEL_NAME)
    mlflow.log_param("max_length", MAX_LENGTH)
    mlflow.log_param("batch_size", 16)
    trainer.train()

    metrics = trainer.evaluate()
    print(metrics)

    mlflow.pytorch.log_model(model, name="model", serialization_format="pickle")

  1%|▏         | 50/3580 [00:14<16:29,  3.57it/s] 

{'loss': 6.691, 'grad_norm': 26.321136474609375, 'learning_rate': 8.8e-06, 'epoch': 0.07}


  3%|▎         | 101/3580 [00:29<16:58,  3.41it/s]

{'loss': 5.6606, 'grad_norm': 45.974056243896484, 'learning_rate': 1.88e-05, 'epoch': 0.14}


  4%|▍         | 151/3580 [00:43<16:44,  3.41it/s]

...

{'train_runtime': 1101.7808, 'train_samples_per_second': 104.045, 'train_steps_per_second': 3.249, 'train_loss': 3.358896694503017, 'epoch': 5.0}


100%|██████████| 154/154 [00:09<00:00, 16.17it/s]


{'eval_loss': 3.6037073135375977, 'eval_type_accuracy': 0.8120928338762216, 'eval_queue_accuracy': 0.4543973941368078, 'eval_category_accuracy': 0.6547231270358306, 'eval_priority_accuracy': 0.5219869706840391, 'eval_runtime': 9.5589, 'eval_samples_per_second': 513.866, 'eval_steps_per_second': 16.111, 'epoch': 4.9965108164689465}


2026/07/09 15:23:00 WARNING mlflow.pytorch: Saving pytorch model by Pickle or CloudPickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is to set `serialization_format` to 'pt2' to save the PyTorch model using the safe graph model format.


PicklingError: Cannot pickle a prepared model with automatic mixed precision, please unwrap the model with `Accelerator.unwrap_model(model)` before pickling it.

In [13]:
import mlflow
import torch

# 1. Az önce başarıyla biten eğitimin Run ID'sini alıyoruz
run_id = mlflow.last_active_run().info.run_id

# 2. O eğitimin içine tekrar bağlanıyoruz
with mlflow.start_run(run_id=run_id):
    unwrapped_model = trainer.accelerator.unwrap_model(model)

    # A) Sadece saf ağırlıkları (tensörleri) geçici bir dosyaya çekiyoruz
    torch.save(unwrapped_model.state_dict(), "temp_clean_weights.pt")

    # B) Hiç Trainer yüzü görmemiş, saf, taze bir model klonu yaratıyoruz
    clean_model = MultiTaskTransformer(MODEL_NAME, num_classes_dict, class_weights)

    # C) Eğitilmiş ağırlıkları bu temiz modele giydiriyoruz
    clean_model.load_state_dict(torch.load("temp_clean_weights.pt"))

    # D) Ve bu pırıl pırıl modeli MLflow'a kaydediyoruz!
    mlflow.pytorch.log_model(clean_model, "model", serialization_format="pickle")
# 3. Model Registry'ye (Staging) aktarıyoruz
model_uri = f"runs:/{run_id}/model"
result = mlflow.register_model(model_uri, "customer_ticket_bert_multitask")

client = mlflow.MlflowClient()
client.set_registered_model_alias("customer_ticket_bert_multitask", "staging", result.version)
print(f"Model kaydedildi: version {result.version}, alias='staging'")

2026/07/09 15:29:33 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/07/09 15:29:34 WARNING mlflow.pytorch: Saving pytorch model by Pickle or CloudPickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is to set `serialization_format` to 'pt2' to save the PyTorch model using the safe graph model format.
2026/07/09 15:29:35 WARNING mlflow.utils.requirements_utils: Found torch version (2.13.0+cu126) contains a local version label (+cu126). MLflow logged a pip requirement for this package as 'torch==2.13.0' without the local version label to make it installable from PyPI. To specify pip requirements containing local version labels, please use `conda_env` or `pip_requirements`.
2026/07/09 15:29:57 WARNING mlflow.utils.environment: Encountered an unexpected error while inferring pip requirements (model 

Model kaydedildi: version 1, alias='staging'


Created version '1' of model 'customer_ticket_bert_multitask'.


In [ ]:
"""
run_id = mlflow.last_active_run().info.run_id
model_uri = f"runs:/{run_id}/model"

result = mlflow.register_model(model_uri, "customer_ticket_bert_multitask")

client = mlflow.MlflowClient()
# NOT: transition_model_version_stage deprecated, alias tabanli yeni API kullanılıyor
client.set_registered_model_alias("customer_ticket_bert_multitask", "staging", result.version)
print(f"Model kaydedildi: version {result.version}, alias='staging'")
"""